In [1]:
library(randomForest)
library(dplyr)

Warning message:
“package ‘randomForest’ was built under R version 4.4.3”
randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.

Warning message:
“package ‘dplyr’ was built under R version 4.4.3”

Attaching package: ‘dplyr’


The following object is masked from ‘package:randomForest’:

    combine


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
# ── 1. Load data ──────────────────────────────────────────────────────────────

african <- read.csv("african_samples.csv", stringsAsFactors = FALSE)
asian   <- read.csv("asian_samples.csv",   stringsAsFactors = FALSE)

cat("African samples:", nrow(african), "\n")
cat("Asian samples:  ", nrow(asian),   "\n\n")


African samples: 1000 
Asian samples:   1000 



In [4]:
head(african)

,id,rifampicin_phenotype,isoniazid_phenotype,pyrazinamide_phenotype,mutations
,<int>,<chr>,<chr>,<chr>,<chr>
1,1,R,R,S,fabG1 c-15t;katG D300F;katG D503R;katG D505G;katG H300G;pncA S129F;rpoB D201F;rpoB D435D;rpoB D435Y;rpoB H270H;rpoB H502E;rpoB S451S
2,2,R,R,S,katG D503R;katG H439D;katG S315T;pncA H300G;rpoB D201F;rpoB D555V;rpoB S129F
3,3,R,R,S,katG D503R;katG H445Y;katG L441V;pncA H300G;pncA L123G;rpoB D201F;rpoB D503R;rpoB D555V;rpoB S450L;rpoB S451S
4,4,R,R,S,katG D555V;pncA H300G;rpoB H501T;rpoB S450L;rpoB S451S
5,5,R,R,S,katG D300F;katG D301R;katG D440L;katG H506Y;katG S315T;pncA D201F;pncA H300G;rpoB D201F;rpoB D555V;rpoB S450L;rpoB S451T
6,6,R,R,S,katG D301R;katG D440L;katG D503R;katG D555V;katG H300G;pncA H270H;rpoB D201F;rpoB D435D;rpoB D555V;rpoB H270H;rpoB H501T


In [5]:
# ── 2. Parse mutation strings into a binary feature matrix ────────────────────
# Each unique "gene mutation" becomes a column; value is 0/1.

parse_mutations <- function(df) {
  # Split each sample's mutation string into a list of tokens
  mut_lists <- strsplit(df$mutations, ";")
  mut_lists <- lapply(mut_lists, trimws)

  # Collect all unique mutations
  all_muts <- sort(unique(unlist(mut_lists)))
  all_muts <- all_muts[all_muts != ""]   # drop empties

  # Build binary matrix
  mat <- matrix(0L, nrow = nrow(df), ncol = length(all_muts),
                dimnames = list(NULL, make.names(all_muts)))

  for (i in seq_along(mut_lists)) {
    cols <- make.names(mut_lists[[i]])
    cols <- cols[cols %in% colnames(mat)]
    if (length(cols)) mat[i, cols] <- 1L
  }

  as.data.frame(mat)
}

In [6]:
african_feats <- parse_mutations(african)
asian_feats   <- parse_mutations(asian)

In [7]:
cat("Feature columns in African data:", ncol(african_feats), "\n")
cat("Feature columns in Asian data:  ", ncol(asian_feats),   "\n\n")


Feature columns in African data: 62 
Feature columns in Asian data:   62 



In [11]:
# ── 3. Align Asian feature columns to African (training) feature set ──────────
# Columns present in Africa but absent in Asia are set to 0.
# Columns present in Asia but absent in Africa are dropped.

align_features <- function(test_feats, train_feats) {
  missing_cols <- setdiff(colnames(train_feats), colnames(test_feats))
  extra_cols   <- setdiff(colnames(test_feats),  colnames(train_feats))
  if (length(missing_cols) > 0) {
    cat("  Mutations in training set but NOT in test set (", length(missing_cols),
        "):\n  ", paste(missing_cols, collapse = ", "), "\n\n", sep = "")
    # Add zero columns for mutations the model was trained on but never saw in Asia
    zero_df <- as.data.frame(matrix(0L, nrow = nrow(test_feats),
                                    ncol = length(missing_cols),
                                    dimnames = list(NULL, missing_cols)))
    test_feats <- cbind(test_feats, zero_df)
  }

  if (length(extra_cols) > 0) {
    cat("  Mutations in test set but NOT in training set (", length(extra_cols),
        ") -- ignored by model:\n  ", paste(extra_cols, collapse = ", "), "\n\n", sep = "")
  }

  test_feats[, colnames(train_feats), drop = FALSE]
}
    

In [12]:
# ── 4. Train one Random Forest per drug and evaluate on Asian data ────────────

drugs <- c("rifampicin", "isoniazid", "pyrazinamide")

results <- list()

for (drug in drugs) {
  cat("══════════════════════════════════════════\n")
  cat("Drug:", toupper(drug), "\n")
  cat("══════════════════════════════════════════\n")

  pheno_col <- paste0(drug, "_phenotype")

  # Phenotype vectors (factor with levels S, R)
  y_train <- factor(african[[pheno_col]], levels = c("S", "R"))
  y_test  <- factor(asian[[pheno_col]],   levels = c("S", "R"))

  cat("Training phenotype distribution:\n")
  print(table(y_train))
  cat("Test phenotype distribution:\n")
  print(table(y_test))
  cat("\n")

  # Align features
  cat("Aligning Asian features to African training features...\n")
  asian_aligned <- align_features(asian_feats, african_feats)

  # Train
  set.seed(123)
  rf <- randomForest(
    x         = african_feats,
    y         = y_train,
    ntree     = 500,
    importance = TRUE
  )

  # Predict on African data (OOB) and Asian data
  pred_train_oob <- rf$predicted          # OOB predictions
  pred_asian      <- predict(rf, newdata = asian_aligned)

  # Confusion matrices
  cat("── OOB confusion matrix (African training data) ──\n")
  print(table(Predicted = pred_train_oob, Actual = y_train))

  cat("\n── Test confusion matrix (Asian data) ──\n")
  cm <- table(Predicted = pred_asian, Actual = y_test)
  print(cm)

  # Accuracy, sensitivity, specificity
  acc  <- sum(diag(cm)) / sum(cm)
  # Sensitivity = TP / (TP + FN)  [correctly calling R]
  if ("R" %in% rownames(cm) && "R" %in% colnames(cm)) {
    sens <- cm["R", "R"] / sum(cm[, "R"])
    spec <- cm["S", "S"] / sum(cm[, "S"])
  } else {
    sens <- NA; spec <- NA
  }

  cat(sprintf("\n  Accuracy:    %.1f%%\n", acc  * 100))
  cat(sprintf("  Sensitivity: %.1f%%  (correctly identifying resistant)\n", sens * 100))
  cat(sprintf("  Specificity: %.1f%%  (correctly identifying susceptible)\n\n", spec * 100))

  # Top 10 most important mutations
  imp <- importance(rf)
  imp_df <- data.frame(
    mutation   = rownames(imp),
    MeanDecreaseGini = imp[, "MeanDecreaseGini"]
  ) %>% arrange(desc(MeanDecreaseGini)) %>% head(10)

  cat("── Top 10 mutations by importance (MeanDecreaseGini) ──\n")
  print(imp_df, row.names = FALSE)
  cat("\n")

  results[[drug]] <- list(model = rf, confusion = cm,
                          accuracy = acc, sensitivity = sens, specificity = spec)
}

cat("══════════════════════════════════════════\n")
cat("Summary\n")
cat("══════════════════════════════════════════\n")
summary_df <- data.frame(
  Drug        = toupper(drugs),
  Accuracy    = sapply(drugs, function(d) sprintf("%.1f%%", results[[d]]$accuracy    * 100)),
  Sensitivity = sapply(drugs, function(d) sprintf("%.1f%%", results[[d]]$sensitivity * 100)),
  Specificity = sapply(drugs, function(d) sprintf("%.1f%%", results[[d]]$specificity * 100))
)
print(summary_df, row.names = FALSE)

══════════════════════════════════════════
Drug: RIFAMPICIN 
══════════════════════════════════════════
Training phenotype distribution:
y_train
  S   R 
500 500 
Test phenotype distribution:
y_test
  S   R 
500 500 

Aligning Asian features to African training features...
── OOB confusion matrix (African training data) ──
         Actual
Predicted   S   R
        S 497 102
        R   3 398

── Test confusion matrix (Asian data) ──
         Actual
Predicted   S   R
        S 499 119
        R   1 381

  Accuracy:    88.0%
  Sensitivity: 76.2%  (correctly identifying resistant)
  Specificity: 99.8%  (correctly identifying susceptible)

── Top 10 mutations by importance (MeanDecreaseGini) ──
    mutation MeanDecreaseGini
  rpoB.S450L       158.123658
  katG.S315T        81.775938
 fabG1.c.15t        14.722106
  rpoB.D555V         6.649177
  katG.D440L         6.400197
  katG.D503R         6.376624
  rpoB.S451S         6.343890
  rpoB.D503R         6.243366
  katG.H300G         6.229691
